# MailMind AI — Exploratory Data Analysis

This notebook explores the synthetic email corpus and the trained classifier.
Run it from the repository root (the cell below puts `src/` on the path).

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / 'src'))  # if running from notebooks/
sys.path.insert(0, str(pathlib.Path.cwd() / 'src'))         # if running from repo root

import pandas as pd
import matplotlib.pyplot as plt
from mailmind.data import generate_dataset, load_dataset
from mailmind import config
pd.set_option('display.max_colwidth', 80)

## 1. Load / generate the dataset

In [ ]:
try:
    df = load_dataset()
except FileNotFoundError:
    df = generate_dataset()
print(df.shape)
df.head()

## 2. Class balance & metadata

In [ ]:
df['label'].value_counts().reindex(config.CATEGORIES).plot.bar(
    color=[config.CATEGORY_COLORS[c] for c in config.CATEGORIES], title='Emails per category')
plt.tight_layout(); plt.show()

df.groupby('label')[['num_links', 'has_attachment']].mean()

## 3. Text length distribution

In [ ]:
df['body_words'] = df['body'].str.split().str.len()
df.boxplot(column='body_words', by='label', rot=45)
plt.title('Body length by category'); plt.suptitle(''); plt.tight_layout(); plt.show()

## 4. NLP signals on a sample email

In [ ]:
from mailmind.nlp import analyze_text
sample = 'URGENT: please approve the invoice today, the payment deadline is tomorrow!'
sig = analyze_text(sample)
print('keywords :', sig.keywords)
print('sentiment:', sig.sentiment)
print('urgency  :', sig.urgency)
print('intent   :', sig.intent)

## 5. Train (small) and evaluate

In [ ]:
from mailmind.data import add_clean_text, train_test_split_frame
from mailmind.ml import MailMindClassifier, evaluate_predictions

d = add_clean_text(df)
train, test = train_test_split_frame(d)
clf = MailMindClassifier().fit(train.to_dict('records'), train['label'].tolist())
pred = clf.predict(test.to_dict('records'))
m = evaluate_predictions(test['label'].tolist(), pred)
print('accuracy :', round(m['accuracy'], 4))
print('f1_macro :', round(m['f1_macro'], 4))

## 6. Run the full agent on one email

In [ ]:
from mailmind.agent import MailMindAgent
from mailmind.schema import Email
agent = MailMindAgent()
insight = agent.process_email(Email(
    subject='FLASH SALE 70% OFF ends tonight',
    body='Use code SAVE70 at checkout. Free shipping this week. Unsubscribe anytime.',
    sender='deals@shop.com', num_links=4))
print('category:', insight.classification.label, round(insight.classification.confidence, 2))
print('priority:', insight.priority.band, insight.priority.score)
print('summary :', insight.summary)
print('actions :', [a.label for a in insight.suggested_actions])